# Week 10 — Data Analysis and Aggregation

**Course:** IT 2012 Unstructured Data  
**Continues from:** Week 8 (Explore) → Week 9 (Clean) → **★ Week 10: Analyse ★**

---

## Pipeline Progress

```
Acquire → Extract → Store → Explore → Clean → ★ Analyse ★ → Search → Visualize → Deploy
```

The data is clean. Now we **answer questions** with it.

This week we learn to:
- **Combine** data from multiple sources (concat, merge, join)
- **Group** and **aggregate** (split-apply-combine with groupby)
- **Reshape** data (pivot, melt)
- **Work with dates** (time series basics)
- **Query relational databases** (MySQL/SQLite with pandas)
- **Build MongoDB aggregation pipelines**

### Data Files

| File | Description | Simulates |
|------|-------------|-----------|
| `cleaned_documents.json` | 60 cleaned documents from Week 9 | MongoDB collection |
| `projects.json` | 6 project records | MySQL table |
| `doc_projects.json` | Document-to-project mapping | MySQL junction table |
| `categories.json` | Category lookup table | MySQL reference table |
| `users.json` | 6 user records | MySQL users table |
| `monthly_stats.csv` | 12 months of processing stats | Time series data |

## 1. Setup and Load Data

In [ ]:
import json
import numpy as np
import pandas as pd

# Display settings
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 120)

print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

In [ ]:
# Load all data sources
with open("data/cleaned_documents.json") as f:
    docs = json.load(f)
df = pd.DataFrame(docs)

with open("data/projects.json") as f:
    projects = pd.DataFrame(json.load(f))

with open("data/doc_projects.json") as f:
    doc_projects = pd.DataFrame(json.load(f))

with open("data/categories.json") as f:
    categories = pd.DataFrame(json.load(f))

with open("data/users.json") as f:
    users = pd.DataFrame(json.load(f))

monthly = pd.read_csv("data/monthly_stats.csv")

print("Data loaded:")
print(f"  documents:    {df.shape}")
print(f"  projects:     {projects.shape}")
print(f"  doc_projects: {doc_projects.shape}")
print(f"  categories:   {categories.shape}")
print(f"  users:        {users.shape}")
print(f"  monthly:      {monthly.shape}")

In [ ]:
# Quick look at the main DataFrame
print("=== Documents (first 5) ===")
print(df[["title", "source_type", "language", "category", "word_count", "confidence"]].head())
print(f"\nColumns: {list(df.columns)}")

## 2. Combining DataFrames with `concat`

`pd.concat()` **stacks** DataFrames — either vertically (adding rows) or horizontally (adding columns).

### When to use `concat`:
- You have data split across multiple files (e.g., one CSV per month)
- You want to append new records to an existing DataFrame
- You need to add new columns from a separate source

In [ ]:
# Simulate: documents arrived in two batches
batch_1 = df.iloc[:35].copy()   # First 35 documents
batch_2 = df.iloc[35:].copy()   # Remaining 25 documents

print(f"Batch 1: {len(batch_1)} documents")
print(f"Batch 2: {len(batch_2)} documents")

# Vertical concat (add rows) — axis=0 (default)
combined = pd.concat([batch_1, batch_2], axis=0, ignore_index=True)
print(f"\nCombined: {len(combined)} documents")
print(f"Same as original? {len(combined) == len(df)}")

In [ ]:
# ⚠️ Common pitfall: duplicate index
# Without ignore_index=True, the index is preserved
bad_concat = pd.concat([batch_1, batch_2])
print("Without ignore_index — duplicate indices:")
print(f"  Index values: {bad_concat.index.tolist()[:5]}...{bad_concat.index.tolist()[-5:]}")
print(f"  Has duplicates: {bad_concat.index.has_duplicates}")

# With ignore_index — clean 0..n index
good_concat = pd.concat([batch_1, batch_2], ignore_index=True)
print(f"\nWith ignore_index:")
print(f"  Index values: {good_concat.index.tolist()[:5]}...{good_concat.index.tolist()[-5:]}")
print(f"  Has duplicates: {good_concat.index.has_duplicates}")

In [ ]:
# Horizontal concat (add columns) — axis=1
# Example: add a "batch_number" column
batch_1["batch"] = 1
batch_2["batch"] = 2
combined_with_batch = pd.concat([batch_1, batch_2], ignore_index=True)
print("Source distribution per batch:")
print(combined_with_batch.groupby("batch")["source_type"].value_counts())

## 3. Merging DataFrames — `merge` and `join`

`pd.merge()` works like SQL `JOIN` — it combines DataFrames based on **matching column values**.

### Merge Types

| Type | SQL Equivalent | Keeps |
|------|---------------|-------|
| `inner` | INNER JOIN | Only rows with matches in **both** |
| `left` | LEFT JOIN | All from left + matches from right |
| `right` | RIGHT JOIN | All from right + matches from left |
| `outer` | FULL OUTER JOIN | All rows from **both** |

In [ ]:
# First, let's look at what we're merging
print("=== doc_projects (junction table) ===")
print(doc_projects.head())
print(f"\n{len(doc_projects)} documents are assigned to projects")
print(f"{len(df)} total documents")
print(f"{len(df) - len(doc_projects)} documents have NO project assignment")

In [ ]:
# INNER JOIN — only documents that have a project assignment
inner = pd.merge(df, doc_projects, left_on="_id", right_on="doc_id", how="inner")
print(f"INNER join: {len(inner)} rows (only docs WITH project)")
print(inner[["title", "project_id", "source_type"]].head())

In [ ]:
# LEFT JOIN — all documents, project info where available
left = pd.merge(df, doc_projects, left_on="_id", right_on="doc_id", how="left")
print(f"LEFT join: {len(left)} rows (all docs, NaN for those without project)")
print(f"\nDocs with project: {left['project_id'].notna().sum()}")
print(f"Docs without project: {left['project_id'].isna().sum()}")
print("\nSample rows without project:")
print(left[left["project_id"].isna()][["title", "source_type", "project_id"]].head())

In [ ]:
# Chain merges: documents → doc_projects → projects (get project names)
docs_with_projects = (
    df
    .merge(doc_projects, left_on="_id", right_on="doc_id", how="left")
    .merge(projects[["project_id", "project_name", "lead"]], on="project_id", how="left")
)

print("Documents with project details:")
print(docs_with_projects[["title", "project_name", "lead"]].dropna().to_string(index=False))

In [ ]:
# Merge with the categories lookup table
df_enriched = df.merge(categories, on="category", how="left")

print("Enriched DataFrame (with category labels and priority):")
print(df_enriched[["title", "category", "category_label", "priority"]].head(10))
print(f"\nShape after merge: {df_enriched.shape}")

### 💡 Key Takeaway

- Use **inner join** when you only want matched rows (e.g., "show me documents that belong to a project")
- Use **left join** when you want to keep everything from the main table (e.g., "show all documents, with project info if available")
- Use **outer join** when you want to see everything, including unmatched rows from both sides

## 4. GroupBy — Split-Apply-Combine

`groupby()` is one of the most powerful pandas operations. The pattern:

1. **Split** — divide data into groups based on a column
2. **Apply** — perform a calculation on each group
3. **Combine** — merge results back into a DataFrame

This is how you answer questions like "What is the average confidence per source type?".

In [ ]:
# Basic groupby: count documents per source type
print("=== Documents per source type ===")
print(df.groupby("source_type").size())

# Or equivalently:
print("\n=== Using value_counts ===")
print(df["source_type"].value_counts())

In [ ]:
# Multiple aggregations on a single column
print("=== Word count statistics by source type ===")
stats = df.groupby("source_type")["word_count"].agg(["count", "mean", "min", "max", "sum"])
stats = stats.round(1)
print(stats)

In [ ]:
# Multiple aggregation functions on multiple columns
agg_result = df.groupby("source_type").agg(
    doc_count=("title", "count"),
    avg_words=("word_count", "mean"),
    avg_confidence=("confidence", "mean"),
    total_words=("word_count", "sum"),
).round(2)

print("=== Multi-column aggregation ===")
print(agg_result)

In [ ]:
# GroupBy with multiple keys
print("=== Documents by source_type AND language ===")
multi_group = df.groupby(["source_type", "language"]).agg(
    count=("title", "count"),
    avg_confidence=("confidence", "mean"),
).round(3)
print(multi_group)

In [ ]:
# GroupBy on the enriched DataFrame (with category labels)
print("=== Documents by category and priority ===")
category_stats = df_enriched.groupby(["priority", "category_label"]).agg(
    count=("title", "count"),
    avg_words=("word_count", "mean"),
).round(1)
print(category_stats)
print(f"\nTotal documents: {category_stats['count'].sum()}")

In [ ]:
# Custom aggregation with apply and lambda
print("=== Custom: percentage of high-confidence docs (>0.90) per source type ===")

def pct_high_confidence(group):
    return (group["confidence"] > 0.90).mean() * 100

result = df.groupby("source_type").apply(pct_high_confidence, include_groups=False)
print(result.round(1).to_string())

## 5. Pivot Tables and Cross-Tabulation

Pivot tables reorganise data into a matrix format — rows become one dimension, columns become another. Think of it as a "2D groupby".

In [ ]:
# Cross-tabulation: source_type × language
print("=== Cross-tab: source type vs language ===")
ct = pd.crosstab(df["source_type"], df["language"], margins=True)
print(ct)

In [ ]:
# Pivot table: average confidence by source_type and language
print("=== Pivot: avg confidence by source_type × language ===")
pvt = df.pivot_table(
    values="confidence",
    index="source_type",
    columns="language",
    aggfunc="mean",
    margins=True,
).round(3)
print(pvt)

In [ ]:
# Pivot table with multiple aggregations
print("=== Pivot: word count stats by source_type × language ===")
pvt2 = df.pivot_table(
    values="word_count",
    index="source_type",
    columns="language",
    aggfunc=["count", "mean", "sum"],
).round(1)
print(pvt2)

In [ ]:
# Cross-tab on the enriched data: priority × source_type
print("=== Cross-tab: priority × source_type ===")
ct2 = pd.crosstab(df_enriched["priority"], df_enriched["source_type"], margins=True)
print(ct2)

## 6. Reshaping Data — `melt` and `pivot`

Sometimes you need to transform data between **wide** and **long** formats.

| Format | Example | Good for |
|--------|---------|----------|
| **Wide** | Columns: pdf_count, image_count, audio_count | Human reading, dashboards |
| **Long** | Columns: source_type, count | Analysis, plotting, groupby |

In [ ]:
# Our monthly_stats has WIDE format (one column per source type)
print("=== Monthly stats (WIDE format) ===")
print(monthly[["month", "pdf_count", "image_count", "audio_count", "web_count"]].head())

In [ ]:
# Melt: WIDE → LONG (unpivot)
monthly_long = monthly.melt(
    id_vars=["month"],
    value_vars=["pdf_count", "image_count", "audio_count", "web_count"],
    var_name="source_type",
    value_name="count",
)

# Clean up the source_type names
monthly_long["source_type"] = monthly_long["source_type"].str.replace("_count", "")

print("=== Monthly stats (LONG format) ===")
print(monthly_long.head(12))
print(f"\nShape: {monthly_long.shape} (was {monthly.shape})")

In [ ]:
# Pivot: LONG → WIDE (reverse of melt)
back_to_wide = monthly_long.pivot(
    index="month",
    columns="source_type",
    values="count",
)
print("=== Back to WIDE format ===")
print(back_to_wide.head())

### When to use which format?

- **Melt (wide → long)** when you need to do `groupby` on the columns that are currently spread out
- **Pivot (long → wide)** when you want a summary table for display or export
- Many plotting libraries (Plotly, seaborn) prefer **long format**

## 7. Time Series Basics

Working with dates is essential for trend analysis. pandas has excellent datetime support.

In [ ]:
# Convert string dates to datetime
df["processing_date"] = pd.to_datetime(df["processing_date"])
monthly["month"] = pd.to_datetime(monthly["month"])

print(f"processing_date dtype: {df['processing_date'].dtype}")
print(f"month dtype: {monthly['month'].dtype}")

In [ ]:
# Extract date components
df["proc_year"] = df["processing_date"].dt.year
df["proc_month"] = df["processing_date"].dt.month
df["proc_month_name"] = df["processing_date"].dt.month_name()
df["proc_weekday"] = df["processing_date"].dt.day_name()

print("=== Date components extracted ===")
print(df[["title", "processing_date", "proc_month_name", "proc_weekday"]].head(10))

In [ ]:
# Documents processed per month
print("=== Documents per month ===")
monthly_docs = df.groupby("proc_month_name").size()
# Reorder by calendar month
month_order = ["September", "October", "November", "December", "January"]
monthly_docs = monthly_docs.reindex([m for m in month_order if m in monthly_docs.index])
print(monthly_docs)

In [ ]:
# Documents per weekday
print("=== Documents per weekday ===")
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_docs = df.groupby("proc_weekday").size()
weekday_docs = weekday_docs.reindex([d for d in weekday_order if d in weekday_docs.index])
print(weekday_docs)

In [ ]:
# Time series analysis on monthly_stats
print("=== Monthly processing trends ===")
monthly_sorted = monthly.sort_values("month")
print(monthly_sorted[["month", "docs_processed", "errors", "avg_confidence", "storage_gb"]].to_string(index=False))

print(f"\nTotal docs processed in 2024: {monthly['docs_processed'].sum():,}")
print(f"Average per month: {monthly['docs_processed'].mean():.0f}")
print(f"Growth (Jan→Dec): {monthly_sorted.iloc[-1]['docs_processed'] - monthly_sorted.iloc[0]['docs_processed']:+d}")

In [ ]:
# Rolling average (smooths out noise)
monthly_sorted["docs_rolling_3m"] = monthly_sorted["docs_processed"].rolling(window=3).mean()
monthly_sorted["errors_rolling_3m"] = monthly_sorted["errors"].rolling(window=3).mean()

print("=== 3-month rolling averages ===")
print(monthly_sorted[["month", "docs_processed", "docs_rolling_3m", "errors", "errors_rolling_3m"]].round(1).to_string(index=False))

## 8. Working with MySQL in Python

In a real pipeline, some data lives in **relational databases** (MySQL, PostgreSQL) while other data is in **MongoDB** or files. You need to combine them.

### Connection Pattern

```python
# Install: pip install pymysql
import pymysql
import pandas as pd

# Connect to MySQL
connection = pymysql.connect(
    host="localhost",
    user="student",
    password="password",
    database="visit_bosnia",
    charset="utf8mb4",
)

# Read directly into pandas
df_users = pd.read_sql("SELECT * FROM users", connection)
df_projects = pd.read_sql("SELECT * FROM projects WHERE budget > 20000", connection)

# Close connection
connection.close()
```

### For this notebook: SQLite simulation

Since we don't have a MySQL server running, we'll use **SQLite** (built into Python) to demonstrate the same patterns. The pandas API is identical — only the connection string changes.

In [ ]:
import sqlite3

# Create an in-memory SQLite database and load our JSON data as tables
conn = sqlite3.connect(":memory:")

# Load DataFrames into SQLite tables (simulating MySQL tables)
projects.to_sql("projects", conn, index=False)
users.to_sql("users", conn, index=False)
doc_projects.to_sql("doc_projects", conn, index=False)

print("Tables created in SQLite:")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables.to_string(index=False))

In [ ]:
# Query with pd.read_sql — same as you'd do with MySQL
print("=== SELECT * FROM projects WHERE budget_km > 20000 ===")
result = pd.read_sql("SELECT * FROM projects WHERE budget_km > 20000", conn)
print(result.to_string(index=False))

In [ ]:
# SQL JOIN — combine tables in the database
print("=== JOIN: documents assigned to projects (with project names) ===")
query = """
SELECT 
    dp.doc_id,
    p.project_name,
    p.lead,
    p.budget_km
FROM doc_projects dp
JOIN projects p ON dp.project_id = p.project_id
ORDER BY p.project_name
"""
result = pd.read_sql(query, conn)
print(result.to_string(index=False))

In [ ]:
# SQL aggregation
print("=== Documents per project (SQL GROUP BY) ===")
query = """
SELECT 
    p.project_name,
    p.lead,
    COUNT(dp.doc_id) as doc_count,
    p.budget_km
FROM projects p
LEFT JOIN doc_projects dp ON p.project_id = dp.project_id
GROUP BY p.project_id
ORDER BY doc_count DESC
"""
result = pd.read_sql(query, conn)
print(result.to_string(index=False))

In [ ]:
# Combine SQL result with our MongoDB-sourced DataFrame
# Step 1: Get project assignments from SQL
sql_assignments = pd.read_sql("""
    SELECT dp.doc_id, p.project_name, p.lead, p.department
    FROM doc_projects dp
    JOIN projects p ON dp.project_id = p.project_id
""", conn)

# Step 2: Merge with our MongoDB-sourced documents DataFrame
combined = df.merge(sql_assignments, left_on="_id", right_on="doc_id", how="left")

print("=== Combined: MongoDB docs + MySQL project info ===")
print(combined[["title", "source_type", "project_name", "lead"]].head(20).to_string(index=False))
print(f"\nDocs with project: {combined['project_name'].notna().sum()}")
print(f"Docs without project: {combined['project_name'].isna().sum()}")

### 💡 MySQL vs SQLite in Practice

| | MySQL (production) | SQLite (this demo) |
|--|----|----|
| Connection | `pymysql.connect(host, user, password, database)` | `sqlite3.connect("file.db")` |
| pandas API | `pd.read_sql(query, connection)` | Identical |
| SQL syntax | Nearly identical | Nearly identical |
| Multi-user | Yes | No (single-user) |
| Setup | Requires server installation | Built into Python |

When you move to a real project, just swap the connection object — all your `pd.read_sql()` calls stay the same.

## 9. MongoDB Aggregation Pipelines

MongoDB has its own aggregation framework — a sequence of **stages** that transform documents.

Since we don't have MongoDB running, I'll show the **code pattern** alongside the pandas equivalent so you can see how they map.

### Pipeline Stages

| MongoDB Stage | pandas Equivalent | Purpose |
|--------------|-------------------|---------|
| `$match` | `df[df["col"] == val]` | Filter rows |
| `$group` | `df.groupby().agg()` | Group and aggregate |
| `$sort` | `df.sort_values()` | Sort results |
| `$project` | `df[["col1", "col2"]]` | Select columns |
| `$unwind` | `df.explode()` | Flatten arrays |
| `$lookup` | `pd.merge()` | Join collections |
| `$limit` | `df.head(n)` | Limit results |

In [ ]:
# MongoDB aggregation: count documents per source_type, sorted descending
# 
# MongoDB syntax (for reference):
# pipeline = [
#     {"$group": {"_id": "$source_type", "count": {"$sum": 1}}},
#     {"$sort": {"count": -1}},
# ]
# result = list(collection.aggregate(pipeline))

# pandas equivalent:
result = (
    df.groupby("source_type")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print("=== $group + $sort: documents per source_type ===")
print(result.to_string(index=False))

In [ ]:
# MongoDB: filter, group, compute averages
#
# pipeline = [
#     {"$match": {"status": "processed", "confidence": {"$gte": 0.85}}},
#     {"$group": {
#         "_id": "$source_type",
#         "avg_words": {"$avg": "$word_count"},
#         "avg_confidence": {"$avg": "$confidence"},
#         "count": {"$sum": 1},
#     }},
#     {"$sort": {"avg_words": -1}},
# ]

# pandas equivalent:
result = (
    df[(df["status"] == "processed") & (df["confidence"] >= 0.85)]
    .groupby("source_type")
    .agg(
        avg_words=("word_count", "mean"),
        avg_confidence=("confidence", "mean"),
        count=("title", "count"),
    )
    .round(2)
    .sort_values("avg_words", ascending=False)
)
print("=== $match + $group + $sort: high-confidence docs stats ===")
print(result)

In [ ]:
# MongoDB: $lookup (join) between collections
#
# pipeline = [
#     {"$lookup": {
#         "from": "projects",
#         "localField": "project_id",
#         "foreignField": "project_id",
#         "as": "project_info",
#     }},
#     {"$unwind": {"path": "$project_info", "preserveNullAndEmptyArrays": True}},
#     {"$project": {
#         "title": 1,
#         "source_type": 1,
#         "project_name": "$project_info.project_name",
#     }},
# ]

# pandas equivalent (we already did this in Section 3):
result = (
    df.merge(doc_projects, left_on="_id", right_on="doc_id", how="left")
    .merge(projects[["project_id", "project_name"]], on="project_id", how="left")
    [["title", "source_type", "project_name"]]
)
print("=== $lookup equivalent: documents with project names ===")
print(result.head(15).to_string(index=False))

In [ ]:
# Complex pipeline: which project has the highest average document confidence?
#
# pipeline = [
#     {"$lookup": {"from": "doc_projects", "localField": "_id", 
#                  "foreignField": "doc_id", "as": "assignments"}},
#     {"$unwind": "$assignments"},
#     {"$lookup": {"from": "projects", "localField": "assignments.project_id",
#                  "foreignField": "project_id", "as": "project"}},
#     {"$unwind": "$project"},
#     {"$group": {
#         "_id": "$project.project_name",
#         "avg_confidence": {"$avg": "$confidence"},
#         "doc_count": {"$sum": 1},
#         "total_words": {"$sum": "$word_count"},
#     }},
#     {"$sort": {"avg_confidence": -1}},
# ]

# pandas equivalent:
result = (
    df.merge(doc_projects, left_on="_id", right_on="doc_id")
    .merge(projects[["project_id", "project_name"]], on="project_id")
    .groupby("project_name")
    .agg(
        avg_confidence=("confidence", "mean"),
        doc_count=("title", "count"),
        total_words=("word_count", "sum"),
    )
    .round(3)
    .sort_values("avg_confidence", ascending=False)
)
print("=== Complex pipeline: project quality metrics ===")
print(result)

## 10. From Analysis to Insight — Answering Real Questions

Let's put everything together and answer meaningful questions about our document collection.

In [ ]:
# Q1: What's our document collection profile?
print("=" * 60)
print("Q1: COLLECTION PROFILE")
print("=" * 60)
print(f"Total documents: {len(df)}")
print(f"Date range: {df['processing_date'].min()} to {df['processing_date'].max()}")
print(f"\nBy source type:")
print(df["source_type"].value_counts().to_string())
print(f"\nBy language:")
print(df["language"].value_counts().to_string())
print(f"\nBy category:")
print(df["category"].value_counts().to_string())
print(f"\nAverage confidence: {df['confidence'].mean():.3f}")
print(f"Total word count: {df['word_count'].sum():,}")

In [ ]:
# Q2: Which categories need the most attention (lowest confidence)?
print("=" * 60)
print("Q2: QUALITY BY CATEGORY (which need improvement?)")
print("=" * 60)
quality = df_enriched.groupby(["category_label", "priority"]).agg(
    count=("title", "count"),
    avg_confidence=("confidence", "mean"),
    min_confidence=("confidence", "min"),
).round(3).sort_values("avg_confidence")

print(quality)
print(f"\n→ Lowest quality: {quality.index[0][0]} "
      f"(avg confidence: {quality.iloc[0]['avg_confidence']:.3f})")

In [ ]:
# Q3: Are OCR/image documents less reliable than PDFs?
print("=" * 60)
print("Q3: CONFIDENCE BY SOURCE TYPE")
print("=" * 60)
conf_stats = df.groupby("source_type")["confidence"].agg(["mean", "std", "min", "max"]).round(3)
print(conf_stats)
print(f"\n→ Image confidence ({conf_stats.loc['image', 'mean']:.3f}) is "
      f"{'lower' if conf_stats.loc['image', 'mean'] < conf_stats.loc['pdf', 'mean'] else 'higher'} "
      f"than PDF ({conf_stats.loc['pdf', 'mean']:.3f})")

In [ ]:
# Q4: Which projects have the most documentation?
print("=" * 60)
print("Q4: PROJECT DOCUMENTATION COVERAGE")
print("=" * 60)
project_docs = (
    df.merge(doc_projects, left_on="_id", right_on="doc_id")
    .merge(projects[["project_id", "project_name", "budget_km"]], on="project_id")
    .groupby(["project_name", "budget_km"])
    .agg(
        doc_count=("title", "count"),
        source_types=("source_type", lambda x: ", ".join(sorted(set(x)))),
    )
    .sort_values("doc_count", ascending=False)
)
print(project_docs.to_string())
print(f"\nDocuments not assigned to any project: {len(df) - len(doc_projects)}")

In [ ]:
# Q5: Processing volume trends — is our pipeline handling more data?
print("=" * 60)
print("Q5: MONTHLY PROCESSING TRENDS")
print("=" * 60)
ms = monthly.sort_values("month")
print(ms[["month", "docs_processed", "storage_gb"]].to_string(index=False))

# Calculate growth rate
first_half = ms.iloc[:6]["docs_processed"].mean()
second_half = ms.iloc[6:]["docs_processed"].mean()
growth = ((second_half - first_half) / first_half) * 100

print(f"\nH1 average: {first_half:.0f} docs/month")
print(f"H2 average: {second_half:.0f} docs/month")
print(f"Growth: {growth:+.1f}%")

## Summary

### What We Learned

| Concept | Key Function | When to Use |
|---------|-------------|-------------|
| **concat** | `pd.concat([df1, df2])` | Stack DataFrames (same columns) |
| **merge** | `pd.merge(df1, df2, on="key")` | SQL-style joins (matching keys) |
| **groupby** | `df.groupby("col").agg(...)` | Split-apply-combine aggregation |
| **pivot_table** | `df.pivot_table(values, index, columns)` | 2D summary tables |
| **crosstab** | `pd.crosstab(df["a"], df["b"])` | Frequency count matrix |
| **melt** | `df.melt(id_vars, value_vars)` | Wide → Long format |
| **pivot** | `df.pivot(index, columns, values)` | Long → Wide format |
| **datetime** | `pd.to_datetime()`, `.dt.month` | Parse and extract date parts |
| **read_sql** | `pd.read_sql(query, conn)` | Query MySQL/SQLite into pandas |
| **MongoDB agg** | `collection.aggregate(pipeline)` | Server-side aggregation |

### Pipeline Update

```
Acquire → Extract → Store → Explore → Clean → ★ Analyse ★ → Search → Visualize → Deploy
                                                    ↑
                                          You are here!
```

### Next Week

**Week 11: Embeddings and Vector Search** — We'll turn our documents into numerical vectors and build semantic search using sentence-transformers and ChromaDB.

---

### 📚 Reading

| Source | Section |
|--------|---------|
| McKinney Ch 8 | Data Wrangling: Join, Combine, and Reshape |
| McKinney Ch 10 | Data Aggregation and Group Operations |
| McKinney Ch 11 | Time Series (11.1–11.3) |
| McKinney Ch 6.4 | Interacting with Databases |
| PyMySQL docs | https://pymysql.readthedocs.io/en/latest/ |
| MongoDB docs | https://www.mongodb.com/docs/manual/core/aggregation-pipeline/ |

---
*IT 2012 — Unstructured Data. International Burch University. Week 10 lab.*